# Time-resolved characterization with `edges`

This notebook shows `TimexLCA.edges_lcia()`, which characterizes a time-explicit inventory with
the [`edges`](https://edges.readthedocs.io) package: every exchange is characterized with the
characterization factor (CF) of the **year in which it occurs**, instead of a single static CF.

The example database has one foreground process, "heat production", that emits 10 kg CO2. That
emission is not instantaneous: a temporal distribution spreads it 40 % / 60 % across two
consecutive calendar years, even though the process itself runs in a single year. This is exactly
the situation `edges_lcia()` is built for:

- **Biosphere flows** are characterized at the **date of the emission**, taken from `bw_timex`'s
  dynamic inventory. A temporal distribution on a biosphere exchange is respected, which the
  static, expanded biosphere matrix cannot represent (it has one undated cell per flow/process).
- **Technosphere flows** are characterized at the **vintage of the consuming process**, because
  `bw_timex` has no dynamic technosphere inventory. This is a documented limitation, not a knob
  (see the last section).

This is **orthogonal** to `TimexLCA.dynamic_lcia()`: `edges` varies the *characterization factor*
with the year of the exchange, while `dynamic_lcia()` varies the *impact* with the time elapsed
since the emission (e.g. radiative forcing decaying after a CH4 pulse). The two can be combined,
but this notebook only shows `edges_lcia()`.

`edges` is an optional extra: `pip install bw_timex[edges]` (Python 3.11/3.12 only, `edges`
requires `<3.13`).

## 1. Build a small example database

The data below is built directly in a scratch Brightway project so this notebook is
self-contained and does not require ecoinvent. It follows the same pattern as `bw_timex`'s own
test fixtures:

- one background process, "electricity production" (dated 2020), and one foreground process,
  "heat production", that consumes some of that electricity and emits CO2 with a temporal
  distribution. These carry sections 2 to 6.
- a methane emission on "heat production", split half in 2024 and half in 2050, used in
  section 5 with a published, year-dependent method.
- a second, independent product system — "car production" consuming primary aluminium from
  China — used in section 7 to characterize a *technosphere* exchange. Its names, reference
  products and locations deliberately match ecoinvent's, because the built-in method used there
  matches on exactly those fields.

In [1]:
import bw2data as bd
import numpy as np
import pandas as pd
from bw_temporalis import TemporalDistribution
from datetime import datetime

project_name = "timex_example_edges_characterization"
if project_name in bd.projects:
    bd.projects.delete_project(project_name)  # making sure to start from scratch
    bd.projects.purge_deleted_directories()

bd.projects.set_current(project_name)

bd.Database("bio").write(
    {
        ("bio", "CO2"): {
            "type": "emission",
            "name": "carbon dioxide",
            "categories": ("air",),
            "unit": "kilogram",
        },
        # The name and category matter: the published method used in section 5 matches
        # biosphere flows by exactly these fields.
        ("bio", "CH4"): {
            "type": "emission",
            "name": "Methane, fossil",
            "categories": ("air",),
            "unit": "kilogram",
        },
    },
)

bd.Database("db_2020").write(
    {
        ("db_2020", "electricity"): {
            "name": "electricity production",
            "location": "CH",
            "reference product": "electricity",
            "unit": "kilowatt hour",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("db_2020", "electricity")},
                {"amount": 2, "type": "biosphere", "input": ("bio", "CO2")},
            ],
        },
        ("db_2020", "aluminium"): {
            # Named like its ecoinvent counterpart so the GeoPolRisk method in section 7
            # matches it on name, reference product and location.
            "name": "aluminium production, primary",
            "location": "CN",
            "reference product": "aluminium, primary",
            "unit": "kilogram",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("db_2020", "aluminium")},
                {"amount": 12, "type": "biosphere", "input": ("bio", "CO2")},
            ],
        },
    }
)

bd.Database("foreground").write(
    {
        ("foreground", "heat"): {
            "name": "heat production",
            "location": "CH",
            "reference product": "heat",
            "unit": "megajoule",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("foreground", "heat")},
                {
                    "amount": 10,
                    "type": "biosphere",
                    "input": ("bio", "CO2"),
                    # 10 kg CO2 split 40 % / 60 % across two consecutive calendar years.
                    "temporal_distribution": TemporalDistribution(
                        date=np.array([0, 366], dtype="timedelta64[D]"),
                        amount=np.array([0.4, 0.6]),
                    ),
                },
                {
                    "amount": 1,
                    "type": "biosphere",
                    "input": ("bio", "CH4"),
                    # 1 kg CH4, half now and half in ~26 years, to span a wide CF range.
                    "temporal_distribution": TemporalDistribution(
                        date=np.array([0, 9497], dtype="timedelta64[D]"),
                        amount=np.array([0.5, 0.5]),
                    ),
                },
                {"amount": 3, "type": "technosphere", "input": ("db_2020", "electricity")},
            ],
        },
        ("foreground", "car"): {
            "name": "car production",
            "location": "CH",
            "reference product": "car",
            "unit": "unit",
            "type": "process",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("foreground", "car")},
                {"amount": 5, "type": "technosphere", "input": ("db_2020", "aluminium")},
            ],
        },
    }
)

bd.Method(("GWP", "example")).write([(("bio", "CO2"), 1)])

17:06:34+0200 [warning  ] Removing project from project timex_example_edges_characterization list, but not deleting data; if you switch to this project again you will have the same data again. To delete data permanently, pass `(..., delete_dir=True)`.


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 14691.08it/s]

17:06:34+0200 [info     ] Vacuuming database            


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 35246.25it/s]

17:06:34+0200 [info     ] Vacuuming database            


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 40721.40it/s]

17:06:34+0200 [info     ] Vacuuming database            


## 2. Build the `TimexLCA` and calculate the time-explicit inventory

`edges_lcia()` needs both the expanded technosphere matrix (the default) and the dynamic
biosphere inventory (also the default), since it reads emission dates from
`TimexLCA.dynamic_inventory`.

In [2]:
from bw_timex import TimexLCA

node = bd.get_node(database="foreground", code="heat")

tlca = TimexLCA(
    demand={node: 1},
    method=("GWP", "example"),
    database_dates={
        "db_2020": datetime.strptime("2020", "%Y"),
        "foreground": "dynamic",
    },
)
tlca.build_timeline(starting_datetime=datetime(2024, 1, 1))
tlca.lci()
tlca.static_lcia()

print(f"static_score: {tlca.static_score}")

2026-09-14 17:06:34.973 | INFO     | bw_timex.timex_lca:__init__:137 - Initializing TimexLCA object...


2026-09-14 17:06:34.974 | INFO     | bw_timex.timex_lca:__init__:154 - Calculating base LCA...


2026-09-14 17:06:34.986 | INFO     | bw_timex.timex_lca:__init__:171 - Collecting node infos...


2026-09-14 17:06:34.987 | INFO     | bw_timex.timex_lca:build_timeline:343 - No edge filter function provided. Skipping all edges in background databases.


2026-09-14 17:06:34.988 | INFO     | bw_timex.timex_lca:build_timeline:364 - Creating activity time mapping...


...

2026-09-14 17:06:34.992 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


2026-09-14 17:06:35.019 | INFO     | bw_timex.timeline_builder:get_weights_for_interpolation_between_nearest_years:630 - Reference date 2024-01-01 00:00:00 is higher than all provided dates. Data will be taken from the closest lower year.


2026-09-14 17:06:35.026 | INFO     | bw_timex.timex_lca:lci:514 - Expanding matrices...


2026-09-14 17:06:35.028 | INFO     | bw_timex.timex_lca:lci:533 - Calculating dynamic inventory...


Starting graph traversal
Calculation count: 1
static_score: 16.0


The static score characterizes the *entire* 10 kg + 3 kWh * 2 kg/kWh background CO2 = 16 kg CO2
at a single CF, ignoring when each kg was actually emitted.

## 3. Define a year-dependent CF and run `edges_lcia()`

`edges` methods are plain dictionaries (or JSON files) of characterization factors. A CF can be a
symbolic expression (`value_expression`) evaluated per scenario year, with the actual numbers
supplied via `parameters` in the shape `{scenario: {parameter: {year: value}}}`. Here CO2 is worth
1.0 in 2024 and 2.0 in 2025, so the CF used depends on *when* the exchange occurs.

In [3]:
edges_method = {
    "name": "example year-dependent CF",
    "version": "1.0",
    "unit": "kg CO2-eq",
    "interpolation": {
        "axis": "scenario_idx",
        "axis_type": "year",
        "method": "linear",
        "extrapolation": "nearest",
    },
    "exchanges": [
        {
            "supplier": {"name": "carbon dioxide", "categories": ["air"], "matrix": "biosphere"},
            "consumer": {"matrix": "technosphere"},
            "value": 1.0,
            "value_expression": "cf_co2",
        }
    ],
}

parameters = {"example": {"cf_co2": {"2024": 1.0, "2025": 2.0}}}

table = tlca.edges_lcia(
    method=edges_method,
    parameters=parameters,
    scenario="example",
    regionalized=False,
)
table

2026-09-14 17:06:35.463 | DEBUG    | bw_timex.edges_lcia:characterize_time_explicit:454 - 2 of 5 exchanges have no characterization factor (CF == 0) and are dropped from the CF table. They carry 5.9% of the total exchange amount.


,supplier,supplier categories,consumer,consumer location,activity,direction,date,year,amount,CF,impact
0,carbon dioxide,"(air,)",electricity production,CH,357904971242438658,biosphere-technosphere,2024-01-01,2024,6.0,1.0,6.0
1,carbon dioxide,"(air,)",heat production,CH,357904971242438659,biosphere-technosphere,2024-01-01,2024,4.0,1.0,4.0
2,carbon dioxide,"(air,)",heat production,CH,357904971242438659,biosphere-technosphere,2025-01-01,2025,6.0,2.0,12.0


## 4. Compare to the static score

Grouping the result by `year` shows the effect directly: the 4 kg emitted in 2024 are worth 4 * 1.0
= 4, the 6 kg emitted in 2025 are worth 6 * 2.0 = 12, for a foreground total of 16.

**This is the point of the feature.** Without emission-date resolution — if the whole 10 kg
foreground emission had been dated to the process's own single timestamp (2024) instead of being
split by the dynamic inventory — every kilogram would be characterized at the 2024 CF of 1.0,
giving 10 kg * 1.0 = **10**, not 16. Resolving *when* each kilogram is actually emitted, rather
than when the process runs, is what turns that counterfactual 10 into the 16 computed above.

The background electricity's 6 kg CO2 has no temporal distribution of its own, so its emission
date is inherited from the consuming process (2024) and it is characterized at CF 1.0, adding 6.
The total, `edges_score`, is therefore 16 + 6 = 22.

Note that `static_score`, printed below, is *also* 16.0 — but that is a coincidence of this small
example, not the same quantity as the foreground's year-resolved total above. `static_score` is
the ordinary static LCIA of the time-explicit inventory under the original Brightway method
`("GWP", "example")` (a constant CF of 1.0 for CO2), summed over *all* 16 kg of CO2 (10 kg
foreground + 6 kg background) with no year-dependent CF involved at all. It lands on 16 here only
because this example's doubled 2025 CF happens to push the foreground's year-resolved total to the
same number as the foreground+background mass total at a flat CF of 1.0 — do not read anything
into "16 == 16" beyond that coincidence. The comparison that actually demonstrates the feature is
the counterfactual **10** above versus the computed **16**; `static_score` (16) versus `edges_score`
(22) instead shows the combined effect of emission-date resolution *and* the year-dependent CF
together.


In [4]:
print(table.groupby("year")["impact"].sum())
print()
print(f"static_score: {tlca.static_score}")
print(f"edges_score:  {tlca.edges_score}")

year
2024    10.0
2025    12.0
Name: impact, dtype: float64

static_score: 16.0
edges_score:  22.0


## 5. Use a built-in, published method

The method above was written by hand to make the mechanism visible. In practice you will usually
reach for one of the methods that ship with `edges`; `get_available_methods()` lists them.

`("Prospective", "GWP100")` suits a time-explicit inventory particularly well. It implements the
scenario- and time-dependent climate metrics of Watanabe & Cherubini (2026), in which the GWP100
of CH4 and N2O is parameterized by IAM scenario and year from 2005 to 2100 (IMAGE, MESSAGE and
REMIND pathways); the remaining greenhouse gases use IPCC AR6 values. No `parameters` argument is
needed here — the year-by-year values already live in the method file, so you only choose a
`scenario`.

In [5]:
from edges import get_available_methods

available = get_available_methods()
print(f"{len(available)} built-in methods available, including:")
for method in sorted(m for m in available if m[0] in ("Prospective", "GeoPolRisk")):
    print("   ", method)

116 built-in methods available, including:
    ('GeoPolRisk', 'paired', '2024')
    ('Prospective', 'GTP100')
    ('Prospective', 'GTP50')
    ('Prospective', 'GWP100')
    ('Prospective', 'GWP20')


In [6]:
table_gwp = tlca.edges_lcia(
    method=("Prospective", "GWP100"),
    scenario="IMAGE_-_SSP2_M_CP",
    regionalized=False,
)

table_gwp[["supplier", "consumer", "date", "year", "amount", "CF", "impact"]]

2026-09-14 17:06:35.573 | DEBUG    | bw_timex.edges_lcia:characterize_time_explicit:454 - 3 of 5 exchanges have no characterization factor (CF == 0) and are dropped from the CF table. They carry 94.1% of the total exchange amount.


,supplier,consumer,date,year,amount,CF,impact
0,"Methane, fossil",heat production,2024-01-01,2024,0.5,26.3,13.15
1,"Methane, fossil",heat production,2050-01-01,2050,0.5,29.0,14.50


The 1 kg of methane was emitted in two halves, 26 years apart, and each half is characterized with
the GWP100 that the chosen scenario assigns to *its own* year: 26.3 in 2024 and 29.0 in 2050. Those
are the method's published values, not something this notebook supplies — characterized statically,
both halves would share a single number.

Only methane appears in the table, and the reason is worth understanding rather than glossing over.
This method does characterize CO2, at a CF of 1.0 — but under ecoinvent's flow names,
`"Carbon dioxide, fossil"` and `"Carbon dioxide, from soil or biomass stock"`. The toy flow built
in section 1 is simply called `"carbon dioxide"`, so no CF matches it, it is characterized at zero,
and the CF table drops it. `edges_score` is therefore the methane contribution alone:
0.5 * 26.3 + 0.5 * 29.0 = 27.65.

That is the everyday failure mode when using a built-in method: a flow whose name does not match
contributes nothing, silently and without an error. `bw_timex` logs a debug line reporting how many
exchanges were dropped for want of a CF (visible above: 3 of 5, carrying 94.1 % of the total
exchange amount — the CO2 mass), and `statistics()` prints `edges`' own view of the coverage: On a real ecoinvent-based inventory the names line up and CO2 is
characterized normally; on hand-built data, check the table before trusting the score.

In [7]:
tlca.edges_lcia_object.statistics()

+----------------------+---------------------------+
|       Activity       |      heat production      |
|     Method name      | ('Prospective', 'GWP100') |
|         Unit         |         kg CO2-eq.        |
|      Data file       |     Prospective_GWP100    |
|    CFs in method     |            514            |
|       CFs used       |             1             |
|   Unique CFs used    |             1             |
|  Exc. characterized  |             1             |
| Exc. uncharacterized |             3             |
+----------------------+---------------------------+


## 6. Compare IAM scenarios

Because the year-dependent values are stored per scenario, the same time-explicit inventory can be
characterized under several IAM pathways by changing one argument. The mapping step is repeated
per call here for clarity; the inventory itself is calculated only once.

In [8]:
scenario_scores = {}
for scenario in ("IMAGE_-_SSP1_VLLO", "IMAGE_-_SSP2_M_CP", "IMAGE_-_SSP2_L"):
    tlca.edges_lcia(
        method=("Prospective", "GWP100"),
        scenario=scenario,
        regionalized=False,
    )
    scenario_scores[scenario] = tlca.edges_score

pd.Series(scenario_scores, name="kg CO2-eq").to_frame()

2026-09-14 17:06:35.667 | DEBUG    | bw_timex.edges_lcia:characterize_time_explicit:454 - 3 of 5 exchanges have no characterization factor (CF == 0) and are dropped from the CF table. They carry 94.1% of the total exchange amount.


2026-09-14 17:06:35.751 | DEBUG    | bw_timex.edges_lcia:characterize_time_explicit:454 - 3 of 5 exchanges have no characterization factor (CF == 0) and are dropped from the CF table. They carry 94.1% of the total exchange amount.


2026-09-14 17:06:35.835 | DEBUG    | bw_timex.edges_lcia:characterize_time_explicit:454 - 3 of 5 exchanges have no characterization factor (CF == 0) and are dropped from the CF table. They carry 94.1% of the total exchange amount.


,kg CO2-eq
IMAGE_-_SSP1_VLLO,30.50
IMAGE_-_SSP2_M_CP,27.65
IMAGE_-_SSP2_L,29.55


The spread between pathways comes entirely from the methane CFs of 2024 and 2050: a low-overshoot
scenario and a delayed-mitigation scenario disagree about how much warming a tonne of methane
causes in a given year. A static LCIA cannot express that difference at all, and a time-explicit
inventory is what makes the 2050 half sensitive to it.

## 7. Characterize a technosphere exchange

`edges` also characterizes *technosphere* exchanges — CFs attached to a product flowing between
two processes, rather than to an elementary flow. `("GeoPolRisk", "paired", "2024")` is the
built-in example: roughly 40 000 supplier-country / consumer-country pairs expressing geopolitical
supply risk, in kg copper-eq.

This needs a product system with a technosphere exchange worth characterizing, so section 7 uses
the second one built above: "car production" (CH) consuming 5 kg of primary aluminium from China.

Two details are worth copying into real work:

- The aluminium process carries a CO2 emission. `build_timeline()` traverses the supply chain by
  impact, so a branch with no impact under the chosen Brightway method is cut off and never
  becomes time-explicit — a technosphere CF would then have nothing to attach to.
- `regionalized` is left at its default `True`, so `edges`' location-mapping cascade runs:
  direct matches first, then aggregate regions, dynamic ("RoW") regions, containing regions and
  finally a global fallback.

In [9]:
from bw_timex import TimexLCA

car_lca = TimexLCA(
    demand={bd.get_node(database="foreground", code="car"): 1},
    method=("GWP", "example"),
    database_dates={
        "db_2020": datetime.strptime("2020", "%Y"),
        "foreground": "dynamic",
    },
)
car_lca.build_timeline(starting_datetime=datetime(2024, 1, 1))
car_lca.lci()

# Loading and matching ~40 000 CF rows takes a few seconds.
table_geopolrisk = car_lca.edges_lcia(method=("GeoPolRisk", "paired", "2024"))

table_geopolrisk[
    ["supplier", "consumer", "consumer location", "direction", "year", "amount", "CF", "impact"]
]

2026-09-14 17:06:35.842 | INFO     | bw_timex.timex_lca:__init__:137 - Initializing TimexLCA object...


2026-09-14 17:06:35.842 | INFO     | bw_timex.timex_lca:__init__:154 - Calculating base LCA...


2026-09-14 17:06:35.848 | INFO     | bw_timex.timex_lca:__init__:171 - Collecting node infos...


2026-09-14 17:06:35.850 | INFO     | bw_timex.timex_lca:build_timeline:343 - No edge filter function provided. Skipping all edges in background databases.


2026-09-14 17:06:35.850 | INFO     | bw_timex.timex_lca:build_timeline:364 - Creating activity time mapping...


...

Starting graph traversal
Calculation count: 1


Method contains 301 duplicate CF matching signature group(s) covering 604 CF entries. These CFs share the same effective CLIPS matching criteria, so only the first matched CF for a given edge will be applied. Examples: indices=[511, 1169] supplier={"categories": null, "classifications": [], "excludes": ["alloy", "liquid", "market"], "location": "GLO", "matrix": "technosphere", "name": "aluminium producti… consumer={"categories": null, "classifications": [], "excludes": [], "location": "BH", "matrix": "technosphere", "name": null, "operator": "equals", "reference product"… values=[0.02225763421839105, 0.0003578840159751286] | indices=[512, 1170] supplier={"categories": null, "classifications": [], "excludes": ["alloy", "liquid", "market"], "location": "GLO", "matrix": "technosphere", "name": "aluminium producti… consumer={"categories": null, "classifications": [], "excludes": [], "location": "BD", "matrix": "technosphere", "name": null, "operator": "equals", "reference product"… values=

2026-09-14 17:06:42.733 | DEBUG    | bw_timex.edges_lcia:_technosphere_entries:648 - 1 technosphere exchanges into a temporal market were skipped to avoid double-counting the physical flow the market passes on.


2026-09-14 17:06:42.734 | WARNING  | bw_timex.edges_lcia:_technosphere_entries:654 - 2 technosphere exchanges have no resolved process time (timestamp 'dynamic') and are not characterized.


,supplier,consumer,consumer location,direction,year,amount,CF,impact
0,"aluminium production, primary",car production,CH,technosphere-technosphere,2024,5.0,8.364799e-07,0.000004


In [10]:
print(f"edges_score: {car_lca.edges_score:.3e} kg copper-eq")
print("mapping strategies applied:")
for strategy in car_lca.edges_lcia_object.applied_strategies:
    print("   ", strategy)

edges_score: 4.182e-06 kg copper-eq
mapping strategies applied:
    map_exchanges
    map_aggregate_locations
    map_dynamic_locations
    map_contained_locations
    map_remaining_locations_to_global


The `direction` column reads `technosphere-technosphere`, marking a CF that was matched between
two processes rather than from a biosphere flow, and `consumer location` shows which side of the
supplier/consumer pair decided the factor: GeoPolRisk gives the CN-to-CH pair a different risk
than, say, CN-to-DE.

The `year` is 2024, the vintage of the consuming process — **not** an emission date. Technosphere
exchanges have no dynamic inventory to take a date from, which is the first limitation below.

## 8. Limitations

- **Technosphere CFs use the consuming process's vintage, not an emission date.** `bw_timex`
  builds a dynamic inventory only for biosphere flows; there is no dynamic technosphere
  inventory. A technosphere characterization factor (e.g. a resource or land-use CF attached to a
  product exchange rather than an elementary flow) is therefore always evaluated at the year of
  the process that consumes it, even if that exchange itself has a temporal distribution.
- **Temporal markets are characterized once, not twice.** `bw_timex` inserts a "temporal market"
  node between a consuming process and its background supplier to blend supply across database
  vintages. That market column carries the supplying commodity's own identity (same name,
  reference product and location), so a technosphere CF pattern that matches the real supplier
  would also match the edge into the market — and again on the market's own downstream edge to
  the consumer, double-counting one physical flow. `edges_lcia()` drops the edges *into* a
  temporal market and characterizes only the market's edge to the consumer, at the consuming
  process's vintage. This changes the technosphere score compared to a naive per-edge
  characterization, so keep it in mind when comparing numbers against a different tool.
- **CF uncertainty (`use_distributions`) is not supported.** Per-year characterization evaluates a
  single characterization matrix per year; a distribution would need a third dimension that this
  code path does not build. Passing `use_distributions=True` to the underlying adapter raises
  `NotImplementedError`.
- **The timeline route is not supported.** `edges_lcia()` requires the expanded, time-explicit
  matrices built by `TimexLCA.lci()` (the default). Calling `TimexLCA.lci(expand_technosphere=False)`
  first and then `edges_lcia()` raises `NotImplementedError`, since that route has no expanded
  technosphere matrix to characterize.